<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/KEEP_ONLY_THE_REQUIRED_LABELS_PREPROCESSING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import requests
from tqdm.auto import tqdm
from google.colab import drive
import requests
import os
from tqdm.auto import tqdm
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

base = "/content/drive/MyDrive/ECG_Project/results"

train_csv = f"{base}/dataset_split/train_split.csv"
val_csv   = f"{base}/dataset_split/validation_split.csv"
test_csv  = f"{base}/dataset_split/test_split.csv"

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

print("TRAIN:", train_df.shape)
print("VAL:", val_df.shape)
print("TEST:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nOMI distribution:")
print(train_df["OMI"].value_counts(dropna=False))

print("\nValidation OMI:")
print(val_df["OMI"].value_counts(dropna=False))



TRAIN: (15275, 28)
VAL: (2685, 28)
TEST: (1995, 6)

Columns:
['Patient_id', 'ecg_row_record', 'ecg_med_record', 'gender', 'age', 'Time_Interval', 'AMI', 'OMI', 'CTO', 'PCI', 'NSTEMI', 'STEMI', 'UA', 'LM', 'PLAD', 'MLAD', 'DLAD', 'DB', 'PLCX', 'MLCX', 'DLCX', 'OM', 'PRCA', 'MRCA', 'DRCA', 'VF_VT', 'Paced', 'Prior_PCI']

OMI distribution:
OMI
0    14290
1      985
Name: count, dtype: int64

Validation OMI:
OMI
0    2519
1     166
Name: count, dtype: int64


In [ ]:
import h5py
import pandas as pd
import numpy as np

base = "/content/drive/MyDrive/ECG_Project/results"

train_csv = f"{base}/dataset_split/train_split.csv"
val_csv   = f"{base}/dataset_split/validation_split.csv"

train_h5 = f"{base}/preprocessing/processed/train_fixed.h5"
val_h5   = f"{base}/preprocessing/processed/validation.h5"

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)


def decode_ids(ids):
    out = []
    for x in ids:
        if isinstance(x, bytes):
            x = x.decode("utf-8")
        out.append(str(x).strip())
    return out


# ==============================
# READ H5 RECORD IDS
# ==============================
with h5py.File(train_h5, "r") as f:
    train_ids = decode_ids(f["record_ids"][:])

with h5py.File(val_h5, "r") as f:
    val_ids = decode_ids(f["record_ids"][:])


print("H5 train IDs:", len(train_ids))
print("H5 val IDs:", len(val_ids))


# ==============================
# NORMALIZE CSV RECORD IDs
# Remove ".dat" from CSV record IDs
train_df["record_id_norm"] = (
    train_df["ecg_row_record"]
    .astype(str)
    .str.replace(".dat", "", regex=False)
    .str.strip()
)

val_df["record_id_norm"] = (
    val_df["ecg_row_record"]
    .astype(str)
    .str.replace(".dat", "", regex=False)
    .str.strip()
)

# H5 IDs are already correct
train_ids_norm = [str(x).strip() for x in train_ids]
val_ids_norm   = [str(x).strip() for x in val_ids]

# Align
train_meta = train_df.set_index("record_id_norm")
val_meta   = val_df.set_index("record_id_norm")

train_aligned = train_meta.reindex(train_ids_norm)
val_aligned   = val_meta.reindex(val_ids_norm)

print("Missing train metadata:", train_aligned["OMI"].isna().sum())
print("Missing val metadata:", val_aligned["OMI"].isna().sum())

print("\nTRAIN OMI:")
print(train_aligned["OMI"].value_counts(dropna=False))

print("\nVAL OMI:")
print(val_aligned["OMI"].value_counts(dropna=False))

H5 train IDs: 15273
H5 val IDs: 2685
Missing train metadata: 0
Missing val metadata: 0

TRAIN OMI:
OMI
0    14288
1      985
Name: count, dtype: int64

VAL OMI:
OMI
0    2519
1     166
Name: count, dtype: int64


In [ ]:
keep_cols = [
    "Patient_id",
    "OMI",
    "STEMI",
    "NSTEMI",
    "CTO",
    "LM",
    "PLAD",
    "MLAD",
    "DLAD",
    "DB",
    "PLCX",
    "MLCX",
    "DLCX",
    "OM",
    "PRCA",
    "MRCA",
    "DRCA"
]

train_omi = train_aligned[keep_cols].copy()
val_omi   = val_aligned[keep_cols].copy()

train_omi.insert(0, "ecg_row_record", train_ids)
val_omi.insert(0, "ecg_row_record", val_ids)

train_omi.to_csv(
    f"{base}/preprocessing/processed/train_omi.csv",
    index=False
)

val_omi.to_csv(
    f"{base}/preprocessing/processed/validation_omi.csv",
    index=False
)

print("Saved successfully.")
print(train_omi.shape)
print(val_omi.shape)

Saved successfully.
(15273, 18)
(2685, 18)


In [ ]:
keep_cols = [
    "Patient_id",
    "OMI",
    "STEMI",
    "NSTEMI",
    "CTO",
    "LM",
    "PLAD",
    "MLAD",
    "DLAD",
    "DB",
    "PLCX",
    "MLCX",
    "DLCX",
    "OM",
    "PRCA",
    "MRCA",
    "DRCA"
]

train_omi = train_aligned[keep_cols].copy()
val_omi   = val_aligned[keep_cols].copy()

train_omi.insert(0, "record_id", train_ids_norm)
val_omi.insert(0, "record_id", val_ids_norm)

train_omi.to_csv(
    f"{base}/preprocessing/processed/train_omi.csv",
    index=False
)

val_omi.to_csv(
    f"{base}/preprocessing/processed/validation_omi.csv",
    index=False
)

print("Train saved:", train_omi.shape)
print("Validation saved:", val_omi.shape)

print("\nTrain OMI distribution:")
print(train_omi["OMI"].value_counts())

print("\nValidation OMI distribution:")
print(val_omi["OMI"].value_counts())

Train saved: (15273, 18)
Validation saved: (2685, 18)

Train OMI distribution:
OMI
0    14288
1      985
Name: count, dtype: int64

Validation OMI distribution:
OMI
0    2519
1     166
Name: count, dtype: int64
